# Analise da Fronteira — Professor (1990-2015) x Extensao (2016-2025)

## Por que adj_close de 2015 != adj_close de 2026

O Yahoo Finance normaliza o Adj Close assim:
**data mais recente = Close** e todos os precos anteriores sao divididos retroativamente
pelo fator `(preco - dividendo) / preco` a cada pagamento de dividendo.

| Base | Coleta | Normalizacao |
|---|---|---|
| Professor (~2015) | adj_close_2015 | dividendos ate dez/2015 aplicados retroativamente |
| Extensao (2025/26) | adj_close_2026 | dividendos ate 2026 aplicados retroativamente |

**Resultado na fronteira:** para tickers que pagaram dividendos de 2016 a 2026,
o primeiro preco da extensao (jan/2016) e **menor** que o ultimo do professor (dez/2015).
Para tickers com **splits pos-2015** (AAPL 4:1, NVDA 40:1, etc.) o salto e ainda maior.

## Este notebook
Analisa diretamente nossas bases (sem downloads externos), calculando o salto real
`(ext[jan4] / prof[dez30] - 1)` para cada ticker na intersecao.

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('../data_bases')
META     = {'year', 'semester', 'period', 'day_in_semester', 'total_days_sem'}
print('Pronto.')

Pronto.


---
## 1 — Carregar bases e identificar intersecao

In [14]:
prof = pd.read_csv(DATA_DIR / 'professor_consolidado.csv', index_col=0, parse_dates=True)
ext  = pd.read_csv(DATA_DIR / 'extensao_2016_2025.csv',   index_col=0, parse_dates=True)
prof.index = pd.to_datetime(prof.index)
ext.index  = pd.to_datetime(ext.index)

tickers_prof = [c for c in prof.columns if c not in META]
tickers_ext  = [c for c in ext.columns  if c not in META]
intersecao   = sorted(set(tickers_prof) & set(tickers_ext))

ultimo_prof  = prof.index[-1]
primeiro_ext = ext.index[ext.index >= '2016-01-01'][0]

print(f'Professor:  {len(tickers_prof)} tickers | ultimo dia: {ultimo_prof.date()}')
print(f'Extensao:   {len(tickers_ext)} tickers | primeiro dia: {primeiro_ext.date()}')
print(f'Intersecao: {len(intersecao)} tickers presentes em ambas')

Professor:  1100 tickers | ultimo dia: 2015-12-30
Extensao:   708 tickers | primeiro dia: 2016-01-04
Intersecao: 465 tickers presentes em ambas


---
## 2 — Calcular o salto na fronteira para cada ticker

**Salto (%) = `(p_ext / p_prof - 1) x 100`**

- Salto = 0%: sem descontinuidade (ideal)
- Salto < 0%: extensao comeca abaixo do fim do professor (artificial — dividendos/splits)
- Salto > 0%: raro, possivel problema de dados

In [15]:
rows = []
for t in intersecao:
    p_prof = prof.loc[ultimo_prof,  t] if (t in prof.columns and pd.notna(prof.loc[ultimo_prof, t]))  else np.nan
    p_ext  = ext.loc[primeiro_ext, t] if (t in ext.columns  and pd.notna(ext.loc[primeiro_ext, t])) else np.nan

    if pd.isna(p_prof) or pd.isna(p_ext) or p_prof <= 0 or p_ext <= 0:
        rows.append({'ticker': t, 'p_prof': p_prof, 'p_ext': p_ext,
                     'salto_pct': np.nan, 'status': 'dados_faltando'})
        continue

    salto = (p_ext / p_prof - 1) * 100
    rows.append({'ticker': t, 'p_prof': round(p_prof, 4), 'p_ext': round(p_ext, 4),
                 'salto_pct': round(salto, 2), 'status': 'ok'})

df = pd.DataFrame(rows)
ok = df[df['status'] == 'ok'].copy()

print(f'Tickers com ambos os precos disponiveis: {len(ok)}')
print(f'Tickers com dados faltando na fronteira:  {(df["status"] == "dados_faltando").sum()}')
print()
print('=== Distribuicao dos saltos ===')
print(f'  Media:    {ok["salto_pct"].mean():>8.2f}%')
print(f'  Mediana:  {ok["salto_pct"].median():>8.2f}%')
print(f'  Min:      {ok["salto_pct"].min():>8.2f}%   ({ok.loc[ok["salto_pct"].idxmin(), "ticker"]})')
print(f'  Max:      {ok["salto_pct"].max():>8.2f}%   ({ok.loc[ok["salto_pct"].idxmax(), "ticker"]})')
print()
print('=== Por faixa ===')
bins = [
    ('Continuo       |salto| <= 2%',       ok['salto_pct'].abs() <= 2),
    ('Pequeno        2% < |s| <= 10%',    (ok['salto_pct'].abs() > 2)   & (ok['salto_pct'].abs() <= 10)),
    ('Moderado       10% < |s| <= 30%',   (ok['salto_pct'].abs() > 10)  & (ok['salto_pct'].abs() <= 30)),
    ('Grande         30% < |s| <= 60%',   (ok['salto_pct'].abs() > 30)  & (ok['salto_pct'].abs() <= 60)),
    ('Critico        |salto| > 60%',       ok['salto_pct'].abs() > 60),
]
for label, mask in bins:
    n = mask.sum()
    print(f'  {label}: {n:>4} tickers ({n/len(ok)*100:4.1f}%)')

Tickers com ambos os precos disponiveis: 452
Tickers com dados faltando na fronteira:  13

=== Distribuicao dos saltos ===
  Media:       24.54%
  Mediana:    -23.26%
  Min:        -98.15%   (CMG)
  Max:      22400.00%   (CHK)

=== Por faixa ===
  Continuo       |salto| <= 2%:   20 tickers ( 4.4%)
  Pequeno        2% < |s| <= 10%:   81 tickers (17.9%)
  Moderado       10% < |s| <= 30%:  201 tickers (44.5%)
  Grande         30% < |s| <= 60%:  113 tickers (25.0%)
  Critico        |salto| > 60%:   37 tickers ( 8.2%)


---
## 3 — Tickers com fronteira continua (salto <= 2%)

Esses sao os tickers onde a concatenacao e praticamente perfeita.

In [16]:
continuos = ok[ok['salto_pct'].abs() <= 2].sort_values('salto_pct', key=abs)
print(f'Tickers com fronteira continua (|salto| <= 2%): {len(continuos)}')
print(f'  Nao pagam dividendos significativos E nao tiveram splits pos-2015')
print()
print(f'  {"Ticker":8} {"Prof(dez30)":>12} {"Ext(jan4)":>12} {"Salto%":>8}')
print('  ' + '-' * 46)
for _, r in continuos.iterrows():
    print(f'  {r["ticker"]:8} {r["p_prof"]:>12.4f} {r["p_ext"]:>12.4f} {r["salto_pct"]:>8.2f}%')

Tickers com fronteira continua (|salto| <= 2%): 20
  Nao pagam dividendos significativos E nao tiveram splits pos-2015

  Ticker    Prof(dez30)    Ext(jan4)   Salto%
  ----------------------------------------------
  APC           48.3800      48.5200     0.29%
  KORS          40.2200      40.3700     0.37%
  FSLR          66.9900      66.7200    -0.40%
  DISCK         25.0600      25.1600     0.40%
  DISCA         26.5600      26.4100    -0.56%
  PVH           73.1500      72.3813    -1.05%
  AKAM          52.9000      52.3100    -1.12%
  FFIV          98.0600      96.9300    -1.15%
  MU            14.1600      13.9759    -1.30%
  NFX           31.8500      32.2700     1.32%
  EMC           25.7200      25.3728    -1.35%
  ESRX          87.8700      86.6300    -1.41%
  URBN          23.0900      22.7500    -1.47%
  DLTR          77.6200      78.8100     1.53%
  GAS           63.8200      62.7350    -1.70%
  TE            26.8200      26.3644    -1.70%
  MHK          189.8900     186.4

---
## 4 — Top 40 piores descontinuidades

In [17]:
splits_conhecidos = {
    'NVDA': '40:1 (2024)', 'CMG': '50:1 (2024)', 'AMZN': '20:1 (2022)',
    'GOOG': '20:1 (2022)', 'GOOGL': '20:1 (2022)', 'AVGO': '10:1 (2024)',
    'LRCX': '10:1 (2024)', 'NFLX': '10:1 (2024)', 'ORLY': '15:1 (2024)',
    'ICE': '5:1 (2024)', 'TSCO': '5:1 (2024)', 'MNST': '6:1 (2024)',
    'NEE': '4:1 (2020)', 'FAST': '4:1 (2021)', 'APH': '4:1 (2023)',
    'CTAS': '4:1 (2022)', 'AAPL': '4:1 (2020)', 'WMT': '3:1 (2024)',
    'SHW': '3:1 (2021)', 'NDAQ': '3:1 (2024)', 'EW': '3:1 (2023)',
    'ISRG': '3:1+3:1', 'AIV': 'spinoff 2020', 'GE': 'spinoff+rsplit',
    'CMCSA': '2:1 (2017)', 'SRE': '2:1 (2023)', 'DD': '2:1 (2019)', 'ETR': '2:1 (2016)',
    'COL': 'adquirida 2018',
}

piores = ok.sort_values('salto_pct').head(40)
print('Top 40 maiores descontinuidades negativas:')
print()
print(f'  {"Ticker":8} {"Prof(dez30)":>12} {"Ext(jan4)":>12} {"Salto%":>10}  Causa')
print('  ' + '-' * 72)
for _, r in piores.iterrows():
    causa = splits_conhecidos.get(r['ticker'], 'div. acumulados 2016-2026')
    print(f'  {r["ticker"]:8} {r["p_prof"]:>12.4f} {r["p_ext"]:>12.4f} {r["salto_pct"]:>10.2f}%  {causa}')

Top 40 maiores descontinuidades negativas:

  Ticker    Prof(dez30)    Ext(jan4)     Salto%  Causa
  ------------------------------------------------------------------------
  CMG          485.7900       8.9762     -98.15%  50:1 (2024)
  NVDA          33.3900       0.7895     -97.64%  40:1 (2024)
  AIV           40.3600       1.6720     -95.86%  spinoff 2020
  AMZN         689.0700      31.8495     -95.38%  20:1 (2022)
  GOOGL        790.3000      37.6607     -95.23%  20:1 (2022)
  GOOG         771.0000      36.7900     -95.23%  20:1 (2022)
  ORLY         257.6600      16.4073     -93.63%  15:1 (2024)
  AVGO         147.3700      10.8539     -92.63%  10:1 (2024)
  LRCX          80.1700       6.7414     -91.59%  10:1 (2024)
  NFLX         116.7100      10.9960     -90.58%  10:1 (2024)
  ISRG         552.5100      60.8211     -88.99%  3:1+3:1
  MNST         150.2400      24.0567     -83.99%  6:1 (2024)
  TSCO          86.2900      14.3267     -83.40%  5:1 (2024)
  ICE          256.9300  

---
## 5 — Tabela completa da intersecao (ordenada por salto)

In [18]:
print(f'Tabela completa — {len(ok)} tickers')
print()
print(f'  {"Ticker":8} {"Prof":>10} {"Ext":>10} {"Salto%":>8}')
print('  ' + '-' * 42)

prev_faixa = None
for _, r in ok.sort_values('salto_pct').iterrows():
    s = r['salto_pct']
    if s < -60 and prev_faixa != 'critico':
        print('  --- CRITICO (< -60%) ---')
        prev_faixa = 'critico'
    elif -60 <= s < -30 and prev_faixa != 'grande':
        print('  --- GRANDE (-60% a -30%) ---')
        prev_faixa = 'grande'
    elif -30 <= s < -10 and prev_faixa != 'moderado':
        print('  --- MODERADO (-30% a -10%) ---')
        prev_faixa = 'moderado'
    elif -10 <= s < -2 and prev_faixa != 'pequeno':
        print('  --- PEQUENO (-10% a -2%) ---')
        prev_faixa = 'pequeno'
    elif abs(s) <= 2 and prev_faixa != 'continuo':
        print('  --- CONTINUO (|salto| <= 2%) ---')
        prev_faixa = 'continuo'
    elif s > 2 and prev_faixa != 'positivo':
        print('  --- POSITIVO (> +2%) [verificar] ---')
        prev_faixa = 'positivo'
    print(f'  {r["ticker"]:8} {r["p_prof"]:>10.4f} {r["p_ext"]:>10.4f} {r["salto_pct"]:>8.2f}%')

Tabela completa — 452 tickers

  Ticker         Prof        Ext   Salto%
  ------------------------------------------
  --- CRITICO (< -60%) ---
  CMG        485.7900     8.9762   -98.15%
  NVDA        33.3900     0.7895   -97.64%
  AIV         40.3600     1.6720   -95.86%
  AMZN       689.0700    31.8495   -95.38%
  GOOGL      790.3000    37.6607   -95.23%
  GOOG       771.0000    36.7900   -95.23%
  ORLY       257.6600    16.4073   -93.63%
  AVGO       147.3700    10.8539   -92.63%
  LRCX        80.1700     6.7414   -91.59%
  NFLX       116.7100    10.9960   -90.58%
  ISRG       552.5100    60.8211   -88.99%
  MNST       150.2400    24.0567   -83.99%
  TSCO        86.2900    14.3267   -83.40%
  ICE        256.9300    44.1245   -82.83%
  FAST        41.1825     7.6197   -81.50%
  NEE        104.9100    19.9099   -81.02%
  CTAS        92.2400    19.9056   -78.42%
  APH         52.9300    11.5506   -78.18%
  AAPL       107.3200    23.7309   -77.89%
  BBBY        48.6100    11.9500   -75

---
## 6 — Tickers com salto positivo (anomalo — verificar)

In [19]:
positivos = ok[ok['salto_pct'] > 2].sort_values('salto_pct', ascending=False)
print(f'Tickers com salto POSITIVO > +2%: {len(positivos)}')
print()
if len(positivos) > 0:
    print(f'  {"Ticker":8} {"Prof(dez30)":>12} {"Ext(jan4)":>12} {"Salto%":>8}')
    print('  ' + '-' * 46)
    for _, r in positivos.iterrows():
        print(f'  {r["ticker"]:8} {r["p_prof"]:>12.4f} {r["p_ext"]:>12.4f} {r["salto_pct"]:>8.2f}%')
else:
    print('  Nenhum — todos os saltos sao <= +2%')

Tickers com salto POSITIVO > +2%: 8

  Ticker    Prof(dez30)    Ext(jan4)   Salto%
  ----------------------------------------------
  CHK            4.4000     990.0000 22400.00%
  GE            31.0500     129.8635   318.24%
  XRX           10.7200      14.1308    31.82%
  SWN            6.3000       7.7100    22.38%
  RRC           22.9400      24.1517     5.28%
  DO            20.8500      21.8500     4.80%
  NBL           32.2200      33.5000     3.97%
  RIG           12.2700      12.5500     2.28%


---
## 7 — Resumo e impacto para o par trading

In [20]:
print('=' * 65)
print('RESUMO')
print('=' * 65)
print()
print(f'Intersecao total:            {len(intersecao)} tickers')
print(f'Com dados em ambos os lados: {len(ok)} tickers')
print()

faixas = [
    ('Continuo      |salto| <= 2%',    ok['salto_pct'].abs() <= 2),
    ('Pequeno       2% < |s| <= 10%', (ok['salto_pct'].abs() > 2)   & (ok['salto_pct'].abs() <= 10)),
    ('Moderado     10% < |s| <= 30%', (ok['salto_pct'].abs() > 10)  & (ok['salto_pct'].abs() <= 30)),
    ('Grande       30% < |s| <= 60%', (ok['salto_pct'].abs() > 30)  & (ok['salto_pct'].abs() <= 60)),
    ('Critico           |s| > 60%',    ok['salto_pct'].abs() > 60),
]
for label, mask in faixas:
    n = mask.sum()
    print(f'  {label}: {n:>4} tickers ({n/len(ok)*100:4.1f}%)')

print()
print('CAUSA DOS SALTOS:')
print('  Dividendos (2016-2026): maioria dos casos — adj_close_2015 > adj_close_2026')
print('  Splits pos-2015:        ~26 tickers — AAPL, NVDA, AMZN, GOOG, etc.')
print('  Sem impacto:            ~22 tickers — sem divs e sem splits pos-2015')
print()
print('IMPACTO NO PAR TRADING SEMESTRAL:')
print('  Pares formados/testados dentro de 1990-2015:  sem impacto')
print('  Pares formados/testados dentro de 2016-2025:  sem impacto')
print('  Calculo cruzando dez/2015 -> jan/2016:        retornos artificialmente distorcidos')

df.to_csv(DATA_DIR / 'external' / 'analise_fronteira.csv', index=False)
print()
print('Salvo: analise_fronteira.csv')

RESUMO

Intersecao total:            465 tickers
Com dados em ambos os lados: 452 tickers

  Continuo      |salto| <= 2%:   20 tickers ( 4.4%)
  Pequeno       2% < |s| <= 10%:   81 tickers (17.9%)
  Moderado     10% < |s| <= 30%:  201 tickers (44.5%)
  Grande       30% < |s| <= 60%:  113 tickers (25.0%)
  Critico           |s| > 60%:   37 tickers ( 8.2%)

CAUSA DOS SALTOS:
  Dividendos (2016-2026): maioria dos casos — adj_close_2015 > adj_close_2026
  Splits pos-2015:        ~26 tickers — AAPL, NVDA, AMZN, GOOG, etc.
  Sem impacto:            ~22 tickers — sem divs e sem splits pos-2015

IMPACTO NO PAR TRADING SEMESTRAL:
  Pares formados/testados dentro de 1990-2015:  sem impacto
  Pares formados/testados dentro de 2016-2025:  sem impacto
  Calculo cruzando dez/2015 -> jan/2016:        retornos artificialmente distorcidos

Salvo: analise_fronteira.csv


---

# Parte 2 — Sobreposição em 2015: validação da normalização adj_close

## Objetivo

Comparar nosso `adj_close` (coletado em 2026) com o `adj_close` do professor (coletado ~dez/2015)
para todo o ano de 2015 — **sem cruzar a fronteira**, apenas dentro do período de sobreposição.

## Hipótese

Se o professor usou `adj_close` do Yahoo coletado em ~dez/2015, então para qualquer ticker:

```
ratio(date) = nosso_adj_close(date) / prof_adj_close(date) = CONSTANTE  ∀ date ∈ 2015
```

A constante = produto dos fatores de dividendo/split acumulados de jan/2016 até 2026.
- **CV do ratio ≈ 0%** → confirma que ambas as séries são adj_close com bases de normalização diferentes
- **CV alto** → as séries não têm a mesma estrutura (problema de dados, ticker reciclado, etc.)

## Metodologia
1. Coletar `adj_close` de 2015 para todos os tickers da interseção: **Tiingo → Yahoo fallback**
2. Salvar em `prices_sobreposição/` (dados de teste, fora do pipeline principal)
3. Calcular ratio diário para cada ticker em cada data de 2015 disponível em ambas as bases
4. Analisar constância do ratio (coeficiente de variação)

In [21]:
import requests
import time
import yfinance as yf

DATA_DIR_P2     = Path('../data_bases')
PRICES_OVERLAP  = DATA_DIR_P2 / 'prices_sobreposição'
PRICES_OVERLAP.mkdir(exist_ok=True)

TIINGO_TOKEN = 'dadfd331f2cb44969b8f7468006d20ad62b13262'
START_2015   = '2015-01-02'
END_2015     = '2015-12-31'

# Reusar bases já carregadas (ou recarregar se sessão nova)
try:
    _ = prof
except NameError:
    META = {'year', 'semester', 'period', 'day_in_semester', 'total_days_sem'}
    prof = pd.read_csv(DATA_DIR_P2 / 'professor_consolidado.csv', index_col=0, parse_dates=True)
    ext  = pd.read_csv(DATA_DIR_P2 / 'extensao_2016_2025.csv',   index_col=0, parse_dates=True)
    tickers_prof = [c for c in prof.columns if c not in META]
    tickers_ext  = [c for c in ext.columns  if c not in META]
    intersecao   = sorted(set(tickers_prof) & set(tickers_ext))

print(f'Diretorio de saída: {PRICES_OVERLAP}')
print(f'Tickers da intersecao: {len(intersecao)}')

Diretorio de saída: ..\data_bases\prices_sobreposição
Tickers da intersecao: 465


---
## P2.1 — Funções de coleta (Tiingo → Yahoo fallback)

In [22]:
TIINGO_TOKEN = 'dadfd331f2cb44969b8f7468006d20ad62b13262'
WIKI_KEY     = '4xiLQjDZiwfFHxH2Mv99'
FMP_KEY      = 'ix1y2xYM1Kdcjj3VTorXyfjjQOVRype6'

# Mapeamento de ticker antigo -> ticker atual para Yahoo Finance
# Yahoo Finance retorna histórico de 2015 quando chamado pelo ticker atual
CURRENT_TICKER = {
    'ABC':   'COR',    # AmerisourceBergen -> Cencora (2023)
    'ADS':   'BFH',    # Alliance Data Systems -> Bread Financial (2022)
    'ANTM':  'ELV',    # Anthem -> Elevance Health (2022)
    'BLL':   'BALL',   # Ball Corporation (ticker change 2020)
    'CBS':   'PARA',   # CBS -> ViacomCBS -> Paramount (2022)
    'COG':   'CTRA',   # Cabot Oil -> Coterra Energy (2021)
    'DISCA': 'WBD',    # Discovery -> Warner Bros Discovery (2022)
    'FB':    'META',   # Facebook -> Meta (2021)
    'GPS':   'GAP',    # The Gap Inc (ticker change 2022)
    'HCP':   'DOC',    # HCP -> Healthpeak -> DOC (2023)
    'IR':    'TT',     # Ingersoll-Rand -> Trane Technologies (2020)
    'LB':    'BBWI',   # L Brands -> Bath & Body Works (2021)
    'MYL':   'VTRS',   # Mylan -> Viatris (2020)
    'PBCT':  'MTB',    # People's United -> M&T Bank (2022)
    'PKI':   'RVTY',   # PerkinElmer -> Revvity (2022)
    'RTN':   'RTX',    # Raytheon -> RTX (2020)
    'SE':    'ENB',    # Spectra Energy -> Enbridge (2017)
    'STI':   'TFC',    # SunTrust -> Truist Financial (2019)
    'SYMC':  'GEN',    # Symantec -> Norton -> Gen Digital (2022)
    'TMK':   'GL',     # Torchmark -> Globe Life (2019)
    'UTX':   'RTX',    # United Technologies -> RTX (2020)
    'VIAB':  'PARA',   # Viacom -> Paramount (2022)
}


def fetch_tiingo_2015(ticker: str) -> pd.Series | None:
    try:
        r = requests.get(
            f'https://api.tiingo.com/tiingo/daily/{ticker}/prices',
            params={'startDate': START_2015, 'endDate': END_2015,
                    'token': TIINGO_TOKEN, 'resampleFreq': 'daily'},
            timeout=15
        )
        if r.status_code != 200:
            return None
        data = r.json()
        if not data or not isinstance(data, list):
            return None
        df_t = pd.DataFrame(data)
        if 'date' not in df_t.columns or 'adjClose' not in df_t.columns:
            return None
        df_t['date'] = pd.to_datetime(df_t['date'], utc=True).dt.tz_localize(None)
        df_t = df_t.set_index('date').sort_index()
        df_t = df_t[(df_t.index >= START_2015) & (df_t.index <= END_2015)]
        if len(df_t) < 100:
            return None
        return df_t['adjClose'].rename(ticker)
    except Exception:
        return None


def fetch_wiki_2015(ticker: str) -> pd.Series | None:
    """NASDAQ Data Link WIKI — adj_close retroativo, cobre ~1962-2018."""
    try:
        r = requests.get(
            'https://data.nasdaq.com/api/v3/datatables/WIKI/PRICES',
            params={'ticker': ticker, 'date.gte': START_2015, 'date.lte': END_2015,
                    'qopts.columns': 'ticker,date,adj_close', 'api_key': WIKI_KEY},
            timeout=20
        )
        if not r.ok:
            return None
        d = r.json()
        if 'datatable' not in d or not d['datatable']['data']:
            return None
        cols = [c['name'] for c in d['datatable']['columns']]
        df_w = pd.DataFrame(d['datatable']['data'], columns=cols)
        df_w['date'] = pd.to_datetime(df_w['date'])
        s = df_w.set_index('date')['adj_close'].sort_index().rename(ticker)
        if len(s) < 100:
            return None
        return s
    except Exception:
        return None


def fetch_fmp_2015(ticker: str) -> pd.Series | None:
    """FMP — dividend-adjusted endpoint."""
    try:
        r = requests.get(
            'https://financialmodelingprep.com/stable/historical-price-eod/dividend-adjusted',
            params={'symbol': ticker, 'from': START_2015, 'to': END_2015, 'apikey': FMP_KEY},
            timeout=15
        )
        if not r.ok or not isinstance(r.json(), list) or not r.json():
            return None
        df_f = pd.DataFrame(r.json())
        df_f['date'] = pd.to_datetime(df_f['date'])
        df_f = df_f[(df_f['date'] >= START_2015) & (df_f['date'] <= END_2015)].sort_values('date')
        if len(df_f) < 100:
            return None
        col = 'adjClose' if 'adjClose' in df_f.columns else 'close'
        return df_f.set_index('date')[col].rename(ticker)
    except Exception:
        return None


def fetch_yahoo_2015(ticker: str, current: str | None = None) -> pd.Series | None:
    """Yahoo Finance auto_adjust=True. Tenta ticker original e, se falhar, current."""
    def _yf(t):
        try:
            h = yf.Ticker(t).history(start=START_2015, end='2016-01-01', auto_adjust=True)
            if h.empty or 'Close' not in h.columns:
                return None
            s = h['Close'].dropna()
            s.index = pd.to_datetime(s.index).tz_localize(None)
            s = s[(s.index >= START_2015) & (s.index <= END_2015)]
            return s.rename(ticker) if len(s) >= 100 else None
        except Exception:
            return None

    result = _yf(ticker)
    if result is None and current and current != ticker:
        result = _yf(current)
    return result


print('Funcoes definidas: fetch_tiingo_2015, fetch_wiki_2015, fetch_fmp_2015, fetch_yahoo_2015')
print(f'CURRENT_TICKER mapeados: {len(CURRENT_TICKER)}')

Funcoes definidas: fetch_tiingo_2015, fetch_wiki_2015, fetch_fmp_2015, fetch_yahoo_2015
CURRENT_TICKER mapeados: 22


---
## P2.2 — Coleta de 2015 para todos os tickers da interseção

Checkpoint automático: tickers já salvos em `prices_sobreposição/` são pulados.

In [23]:
ja_coletados = {f.stem for f in PRICES_OVERLAP.glob('*.csv')}
pendentes    = [t for t in intersecao if t not in ja_coletados]

print(f'Ja no cache: {len(ja_coletados)} tickers')
print(f'A coletar:   {len(pendentes)} tickers')
print()

log_coleta = {}


def _try_yahoo_orig_then_current(ticker):
    """Tenta Yahoo com ticker original; se falhar, tenta current ticker."""
    current = CURRENT_TICKER.get(ticker)
    for t, label in [(ticker, 'yahoo'), (current, f'yahoo({current})')]:
        if not t:
            continue
        try:
            h = yf.Ticker(t).history(start=START_2015, end='2016-01-01', auto_adjust=True)
            if h.empty or 'Close' not in h.columns:
                continue
            s = h['Close'].dropna()
            s.index = pd.to_datetime(s.index).tz_localize(None)
            s = s[(s.index >= START_2015) & (s.index <= END_2015)]
            if len(s) >= 100:
                return s.rename(ticker), label
        except Exception:
            pass
        time.sleep(0.3)
    return None, None


for i, ticker in enumerate(pendentes, 1):
    s, fonte = None, None

    # 1. Tiingo — adjClose, ticker original
    s = fetch_tiingo_2015(ticker)
    if s is not None:
        fonte = 'tiingo'
    time.sleep(0.3)

    # 2. WIKI — adj_close, cobre US delistadas ate 2018
    if s is None:
        s = fetch_wiki_2015(ticker)
        if s is not None:
            fonte = 'wiki'
        time.sleep(0.3)

    # 3. FMP — dividend-adjusted
    if s is None:
        s = fetch_fmp_2015(ticker)
        if s is not None:
            fonte = 'fmp'
        time.sleep(0.3)

    # 4. Yahoo — original ou current ticker
    if s is None:
        s, fonte = _try_yahoo_orig_then_current(ticker)

    if s is not None:
        s.to_csv(PRICES_OVERLAP / f'{ticker}.csv', header=True)
        log_coleta[ticker] = fonte
    else:
        log_coleta[ticker] = 'falhou'

    if i <= 5 or i % 20 == 0 or i == len(pendentes):
        print(f'  [{i:>3}/{len(pendentes)}] {ticker:<8} {log_coleta[ticker]}')

fontes = pd.Series(log_coleta).value_counts()
print()
print('Resumo desta coleta:')
for fonte, cnt in fontes.items():
    print(f'  {fonte}: {cnt}')

Ja no cache: 462 tickers
A coletar:   3 tickers



$CEG: possibly delisted; no price data found  (1d 2015-01-02 -> 2016-01-01) (Yahoo error = "Data doesn't exist for startDate = 1420174800, endDate = 1451624400")


  [  1/3] CEG      falhou


$DELL: possibly delisted; no price data found  (1d 2015-01-02 -> 2016-01-01) (Yahoo error = "Data doesn't exist for startDate = 1420174800, endDate = 1451624400")


  [  2/3] DELL     falhou


$UA: possibly delisted; no price data found  (1d 2015-01-02 -> 2016-01-01) (Yahoo error = "Data doesn't exist for startDate = 1420174800, endDate = 1451624400")


  [  3/3] UA       falhou

Resumo desta coleta:
  falhou: 3


---
## P2.3 — Carregar cache e alinhar com professor (2015)

In [24]:
META_COLS = {'year', 'semester', 'period', 'day_in_semester', 'total_days_sem'}

# Carregar todos os CSVs do cache
overlap_cache = {}
for f in PRICES_OVERLAP.glob('*.csv'):
    try:
        s = pd.read_csv(f, index_col=0, parse_dates=True).squeeze()
        s.index = pd.to_datetime(s.index).tz_localize(None)
        overlap_cache[f.stem] = s
    except Exception:
        pass

# Fatiar professor apenas em 2015
prof_2015 = prof[(prof.index >= '2015-01-01') & (prof.index <= '2015-12-31')]
prof_2015 = prof_2015[[c for c in prof_2015.columns if c not in META_COLS]]

print(f'Cache carregado:  {len(overlap_cache)} series')
print(f'Professor 2015:   {len(prof_2015)} dias x {prof_2015.shape[1]} tickers')
print(f'Datas professor:  {prof_2015.index[0].date()} → {prof_2015.index[-1].date()}')

# Quantos tickers da intersecao têm dados no cache
tickers_com_cache = [t for t in intersecao if t in overlap_cache]
print(f'Tickers intersecao com cache: {len(tickers_com_cache)} / {len(intersecao)}')

Cache carregado:  462 series
Professor 2015:   251 dias x 1100 tickers
Datas professor:  2015-01-02 → 2015-12-30
Tickers intersecao com cache: 462 / 465


---
## P2.4 — Calcular ratio diário e coeficiente de variação por ticker

**Coeficiente de Variação (CV) = std(ratio) / mean(ratio) × 100%**

- CV ≈ 0% → ratio constante → ambas as séries são adj_close com bases diferentes (hipótese confirmada)
- CV alto → inconsistência (possível raw close, ticker reciclado, ou dados de fonte diferente)

In [25]:
rows_ratio = []

# log_coleta pode não existir se a célula de coleta foi pulada (tudo já em cache)
_log = locals().get('log_coleta', {})

for ticker in intersecao:
    if ticker not in overlap_cache or ticker not in prof_2015.columns:
        rows_ratio.append({'ticker': ticker, 'fonte': 'sem_cache', 'n_dias': 0,
                           'ratio_medio': np.nan, 'ratio_min': np.nan, 'ratio_max': np.nan,
                           'cv_pct': np.nan, 'status': 'sem_dados'})
        continue

    s_ours = overlap_cache[ticker].copy()
    s_prof = prof_2015[ticker].dropna()

    if s_ours.empty or s_prof.empty:
        rows_ratio.append({'ticker': ticker, 'fonte': '?', 'n_dias': 0,
                           'ratio_medio': np.nan, 'ratio_min': np.nan, 'ratio_max': np.nan,
                           'cv_pct': np.nan, 'status': 'vazio'})
        continue

    idx_comum = s_ours.index.intersection(s_prof.index)
    if len(idx_comum) < 30:
        rows_ratio.append({'ticker': ticker, 'fonte': '?', 'n_dias': len(idx_comum),
                           'ratio_medio': np.nan, 'ratio_min': np.nan, 'ratio_max': np.nan,
                           'cv_pct': np.nan, 'status': f'poucos_dias_{len(idx_comum)}'})
        continue

    ours  = s_ours.loc[idx_comum]
    profs = s_prof.loc[idx_comum]
    valid = (profs > 0) & (ours > 0)

    if valid.sum() < 30:
        rows_ratio.append({'ticker': ticker, 'fonte': '?', 'n_dias': int(valid.sum()),
                           'ratio_medio': np.nan, 'ratio_min': np.nan, 'ratio_max': np.nan,
                           'cv_pct': np.nan, 'status': 'dados_invalidos'})
        continue

    ratio = ours[valid] / profs[valid]
    m     = float(ratio.mean())
    cv    = float(ratio.std() / m * 100) if m != 0 else np.nan
    fonte = _log.get(ticker, 'cache')

    rows_ratio.append({
        'ticker':      ticker,
        'fonte':       fonte,
        'n_dias':      int(valid.sum()),
        'ratio_medio': round(m, 6),
        'ratio_min':   round(float(ratio.min()), 6),
        'ratio_max':   round(float(ratio.max()), 6),
        'cv_pct':      round(cv, 4),
        'status':      'ok'
    })

df_ratio = pd.DataFrame(rows_ratio)
ok_r = df_ratio[df_ratio['status'] == 'ok'].copy()

print(f'Tickers com ratio calculado: {len(ok_r)}')
print(f'Tickers sem dados / erro:    {(df_ratio["status"] != "ok").sum()}')
print()

THRESH_CV = 2.0
constante = ok_r[ok_r['cv_pct'] < THRESH_CV]
variavel  = ok_r[ok_r['cv_pct'] >= THRESH_CV]

print('=== Distribuição do CV do ratio (2015) ===')
print(f'  (CV próximo de 0% = ratio constante = adj_close com bases diferentes)')
print(f'  Mediana CV:  {ok_r["cv_pct"].median():.4f}%')
print(f'  Media CV:    {ok_r["cv_pct"].mean():.4f}%')
print(f'  Max CV:      {ok_r["cv_pct"].max():.4f}%  ({ok_r.loc[ok_r["cv_pct"].idxmax(), "ticker"]})')
print()
print(f'  Ratio CONSTANTE (CV < {THRESH_CV}%): {len(constante):>4} tickers ({len(constante)/len(ok_r)*100:.1f}%)')
print(f'  Ratio VARIAVEL  (CV >= {THRESH_CV}%): {len(variavel):>4} tickers ({len(variavel)/len(ok_r)*100:.1f}%)')

Tickers com ratio calculado: 460
Tickers sem dados / erro:    5

=== Distribuição do CV do ratio (2015) ===
  (CV próximo de 0% = ratio constante = adj_close com bases diferentes)
  Mediana CV:  0.0002%
  Media CV:    0.1220%
  Max CV:      12.3852%  (VTR)

  Ratio CONSTANTE (CV < 2.0%):  453 tickers (98.5%)
  Ratio VARIAVEL  (CV >= 2.0%):    7 tickers (1.5%)


---
## P2.5 — Tabela completa: ratio médio e CV por ticker

In [26]:
print(f'{"Ticker":8} {"N_dias":>7} {"Ratio_medio":>12} {"Ratio_min":>12} {"Ratio_max":>12} {"CV%":>8}  {"Fonte":8}')
print('-' * 80)

for _, r in ok_r.sort_values('cv_pct', ascending=False).iterrows():
    flag = '  *** VARIAVEL' if r['cv_pct'] >= THRESH_CV else ''
    print(f'{r["ticker"]:<8} {r["n_dias"]:>7} {r["ratio_medio"]:>12.6f} '
          f'{r["ratio_min"]:>12.6f} {r["ratio_max"]:>12.6f} {r["cv_pct"]:>8.4f}%  '
          f'{str(r["fonte"]):<8}{flag}')

out_tabela = DATA_DIR_P2 / 'external' / 'sobreposicao_2015_tabela.csv'
ok_r.sort_values('cv_pct', ascending=False).to_csv(out_tabela, index=False)
print()
print(f'Salvo: {out_tabela}')

Ticker    N_dias  Ratio_medio    Ratio_min    Ratio_max      CV%  Fonte   
--------------------------------------------------------------------------------
VTR          251     0.767319     0.644745     0.840711  12.3852%  cache     *** VARIAVEL
DD           251     0.362303     0.287360     0.435361  10.5709%  cache     *** VARIAVEL
BBBY         251     0.302012     0.230651     0.367141   9.7678%  cache     *** VARIAVEL
JCI          251     0.615712     0.559157     0.703172   5.4556%  cache     *** VARIAVEL
BBT          251     0.540330     0.503550     0.586861   4.7439%  cache     *** VARIAVEL
HBI          251     0.764194     0.709139     0.775008   3.1894%  cache     *** VARIAVEL
MRO          251     0.878396     0.804634     0.968745   2.8879%  cache     *** VARIAVEL
SPGI         251     0.920148     0.867407     0.974586   1.4645%  cache   
TGNA         251     0.509856     0.504371     0.515655   1.1082%  cache   
LB           251     0.913114     0.910522     0.930704   0.72

---
## P2.6 — Tickers com ratio variável (CV ≥ 2%) — investigação

Esses tickers têm ratio não-constante em 2015, indicando possível problema:
- Ticker reciclado pelo Yahoo (empresa diferente em 2015 vs hoje)
- Dados do professor inconsistentes (split não refletido, close em vez de adj_close)
- Fonte de 2015 retornou preço de outra empresa

In [27]:
if len(variavel) > 0:
    print(f'Tickers com CV >= {THRESH_CV}% ({len(variavel)} tickers):')
    print()
    print(f'  {"Ticker":8} {"N_dias":>7} {"Ratio_medio":>12} {"CV%":>8}  {"Fonte":8}')
    print('  ' + '-' * 52)
    for _, r in variavel.sort_values('cv_pct', ascending=False).iterrows():
        print(f'  {r["ticker"]:<8} {r["n_dias"]:>7} {r["ratio_medio"]:>12.6f} '
              f'{r["cv_pct"]:>8.4f}%  {str(r["fonte"]):<8}')
    print()
    print('Interpretação esperada:')
    print('  - CV entre 2-5%: possível split dentro de 2015 (afeta o ratio para datas pré-split)')
    print('  - CV > 10%:     dado suspeito (ticker reciclado ou fonte errada)')
else:
    print(f'Nenhum ticker com CV >= {THRESH_CV}% — ratio essencialmente constante em todos!')

Tickers com CV >= 2.0% (7 tickers):

  Ticker    N_dias  Ratio_medio      CV%  Fonte   
  ----------------------------------------------------
  VTR          251     0.767319  12.3852%  cache   
  DD           251     0.362303  10.5709%  cache   
  BBBY         251     0.302012   9.7678%  cache   
  JCI          251     0.615712   5.4556%  cache   
  BBT          251     0.540330   4.7439%  cache   
  HBI          251     0.764194   3.1894%  cache   
  MRO          251     0.878396   2.8879%  cache   

Interpretação esperada:
  - CV entre 2-5%: possível split dentro de 2015 (afeta o ratio para datas pré-split)
  - CV > 10%:     dado suspeito (ticker reciclado ou fonte errada)


---
## P2.7 — Resumo e salvar CSV

In [28]:
print('=' * 65)
print('RESUMO — Sobreposição 2015')
print('=' * 65)
print()
print(f'Tickers na interseção:           {len(intersecao)}')
print(f'Com ratio calculado (status ok): {len(ok_r)}')
print(f'Sem dados em 2015:               {(df_ratio["status"] == "sem_dados").sum()}')
print(f'Outros erros:                    {((df_ratio["status"] != "ok") & (df_ratio["status"] != "sem_dados")).sum()}')
print()
print(f'Ratio CONSTANTE (CV < {THRESH_CV}%): {len(constante):>4} tickers ({len(constante)/len(ok_r)*100:.1f}%)')
print(f'Ratio VARIAVEL  (CV >= {THRESH_CV}%): {len(variavel):>4} tickers ({len(variavel)/len(ok_r)*100:.1f}%)')
print()
print('INTERPRETAÇÃO:')
print('  Ratio constante → nosso adj_close_2026 e o professor adj_close_2015 são')
print('  a mesma série, ajustada por um fator fixo = dividendos/splits acumulados.')
print('  Isso confirma que o professor usou adj_close do Yahoo Finance.')
print()

# Salvar CSV completo
out_path = DATA_DIR_P2 / 'external' / 'sobreposicao_2015.csv'
df_ratio.to_csv(out_path, index=False)
print(f'Salvo: {out_path}')

RESUMO — Sobreposição 2015

Tickers na interseção:           465
Com ratio calculado (status ok): 460
Sem dados em 2015:               3
Outros erros:                    2

Ratio CONSTANTE (CV < 2.0%):  453 tickers (98.5%)
Ratio VARIAVEL  (CV >= 2.0%):    7 tickers (1.5%)

INTERPRETAÇÃO:
  Ratio constante → nosso adj_close_2026 e o professor adj_close_2015 são
  a mesma série, ajustada por um fator fixo = dividendos/splits acumulados.
  Isso confirma que o professor usou adj_close do Yahoo Finance.

Salvo: ..\data_bases\external\sobreposicao_2015.csv
